# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files
from pathlib import Path

print("Choose your dataset CSV file:")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(file_name)
    print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())
else:
    print('❌ No file uploaded.')

In [ ]:
# @title 3. Automated EDA
from ydata_profiling import ProfileReport

print("Generating Exploratory Data Analysis report...")
profile = ProfileReport(df, title="AutoML Pilot EDA Report", explorative=True)
profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize
import json

# Configuration - These will be automatically updated by the AutoML Pilot app
target_column = 'target_column =' # @param {type:"string"}
task_type = 'task_type =' # @param ["classification", "regression"]
recipient_email = 'recipient_email =' # @param {type:"string"}

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    
    metrics = {
        "Model": str(best_model).split('(')[0],
        "Accuracy": f"{leaderboard.iloc[0]['Accuracy']:.4f}",
        "AUC": f"{leaderboard.iloc[0]['AUC']:.4f}",
        "F1": f"{leaderboard.iloc[0]['F1']:.4f}"
    }

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, html=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')

    metrics = {
        "Model": str(best_model).split('(')[0],
        "R2": f"{leaderboard.iloc[0]['R2']:.4f}",
        "RMSE": f"{leaderboard.iloc[0]['RMSE']:.4f}",
        "MAE": f"{leaderboard.iloc[0]['MAE']:.4f}"
    }

print('✅ Model training complete and saved as best_model.pkl')

In [ ]:
# @title 5. Email Results & Download
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from google.colab import files

def send_email(to_email, metrics_dict):
    # Note: For this to work in Colab, you'd usually need a specialized service
    # or to provide app-specific passwords. Here we provide a template structure.
    print(f"📧 Preparing to send results to {to_email}...")
    
    msg_body = "<h2>AutoML Pilot - Colab Training Results</h2><table border='1'>"
    for k, v in metrics_dict.items():
        msg_body += f"<tr><td><b>{k}</b></td><td>{v}</td></tr>"
    msg_body += "</table>"
    
    print("Metrics Summary:")
    for k, v in metrics_dict.items():
        print(f"- {k}: {v}")
    
    print("\n✅ Results are ready for download below.")

if recipient_email and '@' in recipient_email:
    send_email(recipient_email, metrics)

if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
else:
    print('❌ Model file not found for download.')